#### Environment & Loading

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv('../data/raw/coffee_production_raw.csv', parse_dates=['arrival_date'])

print(f"Engineering features for {len(df)} transactions...")

Engineering features for 589 transactions...


#### Biological Timing (Lagged Weather)

In [2]:
# Monthly Seasonality
df['arrival_month'] = df['arrival_date'].dt.month

# 2. Climate Interaction: The "Hydro-Thermal" Index
df['heat_rain_ratio'] = df['avg_temp_c'] / (df['cum_rainfall_mm'] + 1)

# 3. Peak Harvest Flag
df['is_peak_harvest'] = df['arrival_month'].isin([11, 12, 1, 2]).astype(int)

print("Agronomic interaction features created.")

Agronomic interaction features created.


#### Geospatial & Quality Encoding

In [3]:
# Regional Baseline (Target Encoding)
regional_potential = df.groupby('region')['volume_mt'].transform('mean')
df['regional_yield_baseline'] = regional_potential

# Quality Ranking (Ordinal Encoding)
grade_map = {'Grade 1': 5, 'Grade 2': 4, 'Grade 3': 3, 'Grade 4': 2, 'Grade 5': 1}
df['grade_score'] = df['grade'].map(grade_map)

print("Geospatial and Quality encoding complete.")

Geospatial and Quality encoding complete.


#### Final Feature Selection & Assembly

In [4]:
# Select the 'Clean' features that a model can rely on
features = [
    'altitude', 'cum_rainfall_mm', 'avg_temp_c', 
    'heat_rain_ratio', 'is_peak_harvest', 
    'regional_yield_baseline', 'grade_score'
]
target = 'volume_mt'

# Final processed dataframe
model_ready_df = df[features + [target]]

# Save to processed folder
model_ready_df.to_csv('../data/processed/model_ready_v1.csv', index=False)

print(f"Feature Engineering Complete. Exported {model_ready_df.shape[1]} columns.")
print(f"Features for ML: {features}")

Feature Engineering Complete. Exported 8 columns.
Features for ML: ['altitude', 'cum_rainfall_mm', 'avg_temp_c', 'heat_rain_ratio', 'is_peak_harvest', 'regional_yield_baseline', 'grade_score']


In [5]:
model_ready_df.head()

,altitude,cum_rainfall_mm,avg_temp_c,heat_rain_ratio,is_peak_harvest,regional_yield_baseline,grade_score,volume_mt
0,1750,46.53,18.49,0.389017,1,11.083923,2,4.22
1,1750,46.53,18.49,0.389017,1,11.083923,3,6.72
2,1750,46.53,18.49,0.389017,1,11.083923,5,8.97
3,1750,46.53,18.49,0.389017,1,11.083923,1,18.94
4,1750,46.53,18.49,0.389017,1,11.083923,5,6.10
